In [21]:
import os


import matplotlib.pyplot as plt
import pertpy as pt
from seaborn import clustermap

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage

In [22]:
import warnings

warnings.filterwarnings("ignore")

import logging
import random

import anndata2ri
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pertpy
import rpy2.rinterface_lib.callbacks
import sc_toolbox
import scanpy as sc
import seaborn as sns
from rpy2.robjects import pandas2ri, numpy2ri

from rpy2.robjects import numpy2ri, default_converter
from rpy2.robjects.conversion import localconverter

sc.settings.verbosity = 0
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)


%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [3]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter
# You may still need to import anndata2ri to get its converter, 
# even if you don't call activate()
import anndata2ri 

# Use a local converter instead of activating globally
with localconverter(ro.default_converter + 
                    anndata2ri.converter + 
                    pandas2ri.converter + 
                    numpy2ri.converter) as cv:
    # Now %R / %%R should work within this context
    pass 

In [4]:
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects import conversion, default_converter
import anndata2ri.scipy2ri as scipy2ri

converter = (ro.default_converter
             + numpy2ri.converter
             + pandas2ri.converter
             + scipy2ri.converter
             + anndata2ri.converter)


In [23]:
adata_orig = sc.read_10x_mtx("/Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/inputs/BRI-1456/filtered_feature_bc_matrix")

In [3]:
adata_orig.var

,gene_ids,feature_types
MIR1302-2HG,ENSG00000243485,Gene Expression
FAM138A,ENSG00000237613,Gene Expression
OR4F5,ENSG00000186092,Gene Expression
AL627309.1,ENSG00000238009,Gene Expression
AL627309.3,ENSG00000239945,Gene Expression
...,...,...
AC141272.1,ENSG00000277836,Gene Expression
AC023491.2,ENSG00000278633,Gene Expression
AC007325.1,ENSG00000276017,Gene Expression
AC007325.4,ENSG00000278817,Gene Expression


In [24]:
INPUT_DIR = "inputs"
entrez_gene_ids = pd.read_csv(os.path.join(INPUT_DIR, "gProfiler_hsapiens_2026-04-29_19-37-55.csv"))
entrez_gene_ids.head(10)

,initial_alias,converted_alias,name,description,namespace
0,ENSG00000238009,NaN,NaN,NaN,NaN
1,ENSG00000239945,NaN,NaN,NaN,NaN
2,ENSG00000241860,NaN,NaN,NaN,NaN
3,ENSG00000286448,NaN,NaN,NaN,NaN
4,ENSG00000237491,NaN,NaN,NaN,NaN
5,ENSG00000177757,400728.0,FAM87B,family with sequence similarity 87 member B [S...,"ARRAYEXPRESS,ENSG"
6,ENSG00000228794,643837.0,LINC01128,long intergenic non-protein coding RNA 1128 [S...,"ARRAYEXPRESS,ENSG"
7,ENSG00000225880,NaN,NaN,NaN,NaN
8,ENSG00000230368,284593.0,FAM41C,family with sequence similarity 41 member C [S...,"ARRAYEXPRESS,ENSG"
9,ENSG00000230699,NaN,NaN,NaN,NaN


In [25]:
entrez_gene_ids = entrez_gene_ids.sort_values("converted_alias").drop_duplicates("initial_alias", keep="first")

In [26]:
entrez_gene_ids.value_counts("converted_alias")

converted_alias
101927815.0    2
100652739.0    2
653720.0       2
653145.0       2
643836.0       2
              ..
28.0           1
29.0           1
30.0           1
31.0           1
131675794.0    1
Name: count, Length: 19368, dtype: int64

In [27]:
entrez_gene_ids.set_index("initial_alias", inplace=True)

In [28]:
RES_DIR = "/Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/DE/subq/"

# Prepare gene sets for fgsea

In [18]:
%%R
library(biomaRt)
library(fgsea)
library(msigdb)
library(dplyr)
library(ggplot2)


Attaching package: ‘dplyr’

The following object is masked from ‘package:biomaRt’:

    select

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



In [19]:
%%R

msigdb_gmt_fp <- "/Users/willtrim/Documents/reference/msigdb/msigdb_v2025.1.Hs_GMTs/"

### Reactome

gmt.file <- paste(msigdb_gmt_fp, "c2.cp.reactome.v2025.1.Hs.entrez.gmt", sep="")
pathways <- gmtPathways(gmt.file)
str(head(pathways))

###  Hallmark 

hallmark.gmt.file <- paste(msigdb_gmt_fp, "h.all.v2025.1.Hs.entrez.gmt", sep="")
hallmark.pathways <- gmtPathways(hallmark.gmt.file)
str(head(hallmark.pathways))

### Wikipathways
wiki.gmt.file <- paste(msigdb_gmt_fp, "c2.cp.wikipathways.v2025.1.Hs.entrez.gmt", sep="")
wiki.pathways <- gmtPathways(wiki.gmt.file)
str(head(wiki.pathways))


### KEGG

library(clusterProfiler)

kegg_pathways_df <- download_KEGG(species = "hsa", keggType = "KEGG", keyType = "kegg")

kegg_pathways_list <- kegg_pathways_df["KEGGPATHID2EXTID"][[1]] %>%
  group_by(from) %>%
  group_split()

library(purrr)
kegg_pathways <- map(kegg_pathways_list, ~pull(.x, to))
kegg_pathway_ids <-  map(kegg_pathways_list, ~ .x[[1, 1]])
kegg_pathways_names_map <- kegg_pathways_df[["KEGGPATHID2NAME"]]$to
names(kegg_pathways_names_map) <- kegg_pathways_df[["KEGGPATHID2NAME"]]$from
kegg_pathways_names <- lapply(kegg_pathway_ids, function(x) {kegg_pathways_names_map[[x]]})
names(kegg_pathways) <- kegg_pathways_names

# ### GTRD

# gtrd.gmt.file <- paste(msigdb_gmt_fp, "c3.tft.gtrd.v2025.1.Hs.entrez.gmt", sep="")
# gtrd.pathways <- gmtPathways(gtrd.gmt.file)
# str(head(gtrd.pathways))


### GO

#### BP
bp.gmt.file <- paste(msigdb_gmt_fp, "c5.go.bp.v2025.1.Hs.entrez.gmt", sep="")
bp.pathways <- gmtPathways(bp.gmt.file)
str(head(bp.pathways))


#### CC
cc.gmt.file <- paste(msigdb_gmt_fp, "c5.go.cc.v2025.1.Hs.entrez.gmt", sep="")
cc.pathways <- gmtPathways(cc.gmt.file)
str(head(cc.pathways))


#### MF

mf.gmt.file <- paste(msigdb_gmt_fp, "c5.go.mf.v2025.1.Hs.entrez.gmt", sep="")
mf.pathways <- gmtPathways(mf.gmt.file)
str(head(mf.pathways))

List of 6
 $ REACTOME_2_LTR_CIRCLE_FORMATION                : chr [1:7] "8815" "3159" "3981" "11168" ...
 $ REACTOME_ABACAVIR_ADME                         : chr [1:9] "5243" "9429" "124" "161823" ...
 $ REACTOME_ABACAVIR_TRANSMEMBRANE_TRANSPORT      : chr [1:5] "5243" "9429" "6580" "6582" ...
 $ REACTOME_ABC_FAMILY_PROTEINS_MEDIATED_TRANSPORT: chr [1:90] "10349" "26154" "20" "21" ...
 $ REACTOME_ABC_TRANSPORTERS_IN_LIPID_HOMEOSTASIS : chr [1:18] "10349" "26154" "20" "21" ...
 $ REACTOME_ABC_TRANSPORTER_DISORDERS             : chr [1:65] "19" "26154" "21" "8647" ...
List of 6
 $ HALLMARK_ADIPOGENESIS       : chr [1:200] "19" "11194" "10449" "33" ...
 $ HALLMARK_ALLOGRAFT_REJECTION: chr [1:200] "16" "6059" "10006" "43" ...
 $ HALLMARK_ANDROGEN_RESPONSE  : chr [1:101] "10257" "11057" "2181" "87" ...
 $ HALLMARK_ANGIOGENESIS       : chr [1:36] "350" "351" "894" "1281" ...
 $ HALLMARK_APICAL_JUNCTION    : chr [1:200] "58" "60" "70" "71" ...
 $ HALLMARK_APICAL_SURFACE     : chr [1:44] "102" 


clusterProfiler v4.16.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

G Yu. Thirteen years of clusterProfiler. The Innovation. 2024,
5(6):100722

Attaching package: ‘clusterProfiler’

The following object is masked from ‘package:biomaRt’:

    select

The following object is masked from ‘package:stats’:

    filter

Reading KEGG annotation online: "https://rest.kegg.jp/link/hsa/pathway"...
Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/hsa"...

Attaching package: ‘purrr’

The following object is masked from ‘package:clusterProfiler’:

    simplify



# By cell_type_

In [9]:
cell_type_col = "cell_type_merged"
gene_id_col = "entrez_id"

## MAST

In [29]:
cell_type_col = "cell_type2_"

In [30]:
de_res_mast = pd.read_csv(
    os.path.join(
        RES_DIR,
        f"MAST_results_per_{cell_type_col}.csv"),
    index_col=0
)
de_res_mast.head()

,Pr(>Chisq),coef,FDR,cell_type,score,ensembl_id,entrez_id
primerid,,,,,,,
HLA-DRB5,4.967041e-27,-4.108800,3.044300e-23,VECs,-22.516513,ENSG00000198502,3127.0
MTRNR2L8,4.013079e-10,2.608595,1.229808e-06,VECs,5.910163,ENSG00000255823,NaN
CYGB,1.770228e-09,2.225677,3.616577e-06,VECs,5.441702,ENSG00000161544,114757.0
RPL28,2.821224e-09,-0.215941,4.322821e-06,VECs,-5.364233,ENSG00000108107,6158.0
FABP5,2.915840e-08,0.581426,3.574236e-05,VECs,4.446817,ENSG00000164687,2171.0


In [31]:
de_res_mast["score"] = -np.log10(de_res_mast["FDR"])
de_res_mast.loc[np.isinf(de_res_mast.score), "score"] = 300
de_res_mast.loc[de_res_mast.coef < 0, "score"] = -de_res_mast.loc[de_res_mast.coef < 0, "score"]

In [32]:
de_res_mast["ensembl_id"] = adata_orig.var.loc[de_res_mast.primerid, "gene_ids"].values

AttributeError: 'DataFrame' object has no attribute 'primerid'

In [ ]:
de_res_mast["entrez_id"] = None
idx = de_res_mast["ensembl_id"].isin(entrez_gene_ids.index)
de_res_mast.loc[idx, "entrez_id"] = entrez_gene_ids.loc[de_res_mast.loc[idx, "ensembl_id"], "converted_alias"].values

In [20]:
de_res_mast.to_csv(os.path.join(
        RES_DIR,
        f"MAST_results_per_{cell_type_col}.csv"),
    index=False
                  )

In [34]:
RES_DIR

'/Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/results/DE/subq/'

In [24]:
de_res_mast_by_ct = dict(list(de_res_mast.groupby('cell_type')))

In [32]:
cell_type_de = de_res_mast_by_ct["B_cells"]

In [34]:
vc = cell_type_de.value_counts(gene_id_col)
dups = vc[vc > 1].index

cell_type_de = cell_type_de.loc[~(cell_type_de[gene_id_col].isin(dups) | cell_type_de[gene_id_col].isna()),:]
cell_type_de[gene_id_col] = cell_type_de[gene_id_col].astype(int).astype(str)
cell_type_de.set_index(gene_id_col, inplace=True)

In [35]:
cell_type_de

,primerid,Pr(>Chisq),coef,FDR,cell_type,score,ensembl_id
entrez_id,,,,,,,
23347,SMCHD1,4.910177e-14,-1.915925,1.549652e-10,B_cells,-9.809766,ENSG00000101596
3117,HLA-DQA2,3.363299e-13,1.943674,5.307285e-10,B_cells,9.275128,ENSG00000237541
7430,EZR,6.886382e-13,-1.953955,7.244474e-10,B_cells,-9.139993,ENSG00000092820
283131,NEAT1,1.738492e-12,-1.770157,1.371670e-09,B_cells,-8.862750,ENSG00000245532
3123,HLA-DRB1,2.675090e-12,-0.898529,1.688517e-09,B_cells,-8.772495,ENSG00000196126
...,...,...,...,...,...,...,...
4713,NDUFB7,5.848669e-04,0.648215,9.322424e-03,B_cells,2.030471,ENSG00000099795
23564,DDAH2,6.038331e-04,0.594151,9.565419e-03,B_cells,2.019296,ENSG00000213722
56829,ZC3HAV1,6.084570e-04,-0.847056,9.565419e-03,B_cells,-2.019296,ENSG00000105939


In [27]:
out_dir = os.path.join(RES_DIR, "fgsea", cell_type_col)
os.makedirs(out_dir, exist_ok=True)

In [26]:
%%R
plot_fgsea_top <- function(fgseaRes, pathways, ranks, n_top) {
    topPathwaysUp <- fgseaRes[ES > 0][head(order(pval), n=n_top), pathway]
    topPathwaysDown <- fgseaRes[ES < 0][head(order(pval), n=n_top), pathway]
    topPathways <- c(topPathwaysUp, rev(topPathwaysDown))
    plotGseaTable(pathways[topPathways], ranks, fgseaRes, gseaParam = 0.5)
}

In [ ]:
for cell_type, cell_type_de in de_res_mast_by_ct.items():
    
    with conversion.localconverter(default_converter + numpy2ri.converter
         + anndata2ri.converter):
        ro.globalenv['ranks'] = cell_type_de.loc[:,"score"]
    
        %R -i out_dir setwd(out_dir)
        
        %R -i cell_type fgseaRes_hallmark <- fgsea(pathways = hallmark.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_hallmark, paste0(cell_type, ".hallmark.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_react <- fgsea(pathways = pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_react, paste0(cell_type, ".reactome.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_wp <- fgsea(pathways = wiki.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_wp, paste0(cell_type, ".wiki.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_kegg <- fgsea(pathways = kegg_pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_kegg, paste0(cell_type, ".kegg.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_bp <- fgsea(pathways = bp.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_bp, paste0(cell_type, ".bp.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_cc <- fgsea(pathways = cc.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_cc, paste0(cell_type, ".cc.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_mf <- fgsea(pathways = mf.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R write.csv(fgseaRes_mf, paste0(cell_type, ".mf.csv"), row.names = FALSE)
    
        %R n_top <- 15
        %R p1 <- plot_fgsea_top(fgseaRes_hallmark, hallmark.pathways, ranks, n_top)
        %R p2 <- plot_fgsea_top(fgseaRes_react, react.pathways, ranks, n_top)
        %R p3 <- plot_fgsea_top(fgseaRes_wp, wiki.pathways, ranks, n_top)
        %R p4 <- plot_fgsea_top(fgseaRes_kegg, kegg_pathways, ranks, n_top)
        %R p5 <- plot_fgsea_top(fgseaRes_bp, bp.pathways, ranks, n_top)
        
    
        ro.r('''
        library(grid)
        
        plots <- list(p1, p2, p3, p4, p5)
        
        pdf(paste0(cell_type, ".pdf"), width = 16, height = 16)
        for (i in seq_along(plots)) {
            # if (i > 1) grid.newpage()
            grid.draw(plots[[i]])
        }
        dev.off()
        ''')

## DeSeq2

In [120]:
cell_type_col = "cell_type_"

In [121]:
de_res = pd.read_csv(
    os.path.join(
        RES_DIR,
        f"deseq2_results_per_{cell_type_col}.csv"),
    index_col=0
)
de_res.head()

,variable,baseMean,log_fc,lfcSE,stat,p_value,adj_p_value,contrast,cell_type
0,CFD,52.585499,2.528972,0.651973,3.878953,0.000105,0.656954,NaN,ECs
1,CLU,170.926114,-1.977524,0.527137,-3.751442,0.000176,0.656954,NaN,ECs
2,CYGB,15.959097,2.504661,0.721325,3.472304,0.000516,0.999743,NaN,ECs
3,IGFBP3,29.914866,2.273218,0.726247,3.130092,0.001748,0.999743,NaN,ECs
4,ICAM1,250.555577,-0.930265,0.319276,-2.913676,0.003572,0.999743,NaN,ECs


In [122]:
de_res["ensembl_id"] = adata_orig.var.loc[de_res.variable, "gene_ids"].values

In [123]:
de_res["entrez_id"] = None
idx = de_res["ensembl_id"].isin(entrez_gene_ids.index)
de_res.loc[idx, "entrez_id"] = entrez_gene_ids.loc[de_res.loc[idx, "ensembl_id"], "converted_alias"].values

In [124]:
de_res.to_csv(
    os.path.join(
        RES_DIR,
        f"deseq2_results_per_{cell_type_col}.csv"),
    index=False
)

In [39]:
de_res_by_ct = dict(list(de_res.groupby('cell_type')))

In [40]:
de_res_by_ct["Mesothelial"]

,variable,baseMean,log_fc,lfcSE,stat,p_value,adj_p_value,contrast,cell_type,ensembl_id,entrez_id
0,LYZ,39.173049,-1.648288,0.269394,-6.118509,9.445466e-10,0.000010,NaN,Mesothelial,ENSG00000090382,4069.0
1,GZMB,21.675012,-2.226017,0.365557,-6.089381,1.133483e-09,0.000010,NaN,Mesothelial,ENSG00000100453,3002.0
2,IGLC2,32.244079,1.969933,0.334969,5.880938,4.079490e-09,0.000023,NaN,Mesothelial,ENSG00000211677,NaN
3,PDZK1IP1,418.916476,0.906541,0.169227,5.356952,8.463775e-08,0.000364,NaN,Mesothelial,ENSG00000162366,10158.0
4,MPRIP-AS1,24.830294,-2.018823,0.389383,-5.184673,2.163944e-07,0.000745,NaN,Mesothelial,ENSG00000225442,NaN
...,...,...,...,...,...,...,...,...,...,...,...
17571,PCK1,447.314482,1.889813,1.339761,1.410560,NaN,NaN,NaN,Mesothelial,ENSG00000124253,5105.0
17572,CLDN5,14.775053,2.425018,1.585904,1.529107,NaN,NaN,NaN,Mesothelial,ENSG00000184113,7122.0
17573,HMOX1,107.798478,2.703812,1.232615,2.193558,NaN,NaN,NaN,Mesothelial,ENSG00000100292,3162.0
17574,PWWP3B,171.176119,1.611988,0.834273,1.932206,NaN,NaN,NaN,Mesothelial,ENSG00000157502,139221.0


In [46]:
out_dir = os.path.join(RES_DIR, "fgsea", cell_type_col, "deseq2")
os.makedirs(out_dir, exist_ok=True)

In [50]:
%%R
library(data.table)

plot_fgsea_top <- function(fgseaRes, pathways, ranks, n_top) {
    topPathwaysUp <- fgseaRes[ES > 0][head(order(pval), n=n_top), pathway]
    topPathwaysDown <- fgseaRes[ES < 0][head(order(pval), n=n_top), pathway]
    topPathways <- c(topPathwaysUp, rev(topPathwaysDown))
    plotGseaTable(pathways[topPathways], ranks, fgseaRes, gseaParam = 0.5)
}

data.table 1.17.8 using 1 threads (see ?getDTthreads).  Latest news: r-datatable.com
**********
This installation of data.table has not detected OpenMP support. It should still work but in single-threaded mode.
This is a Mac. Please read https://mac.r-project.org/openmp/. Please engage with Apple and ask them for support. Check r-datatable.com for updates, and our Mac instructions here: https://github.com/Rdatatable/data.table/wiki/Installation. After several years of many reports of installation problems on Mac, it's time to gingerly point out that there have been no similar problems on Windows or Linux.
**********

Attaching package: ‘data.table’

The following object is masked from ‘package:purrr’:

    transpose

The following objects are masked from ‘package:dplyr’:

    between, first, last



In [65]:
cell_type

'Myeloid_cells'

In [54]:
cell_type_de.loc[:,"score"]

entrez_id
3123    -5.901214
4069    -5.548287
51065    5.394693
23114   -5.283129
6279    -4.867909
           ...   
3778    -0.447592
6490     1.161309
9244     0.215611
3929     1.914940
30848    1.487380
Name: score, Length: 14663, dtype: float64

In [56]:
cell_type_de = de_res_by_ct["FAPs"]

In [57]:
cell_type_de["score"] = cell_type_de["stat"]
vc = cell_type_de.value_counts(gene_id_col)
dups = vc[vc > 1].index


In [60]:
cell_type_de = cell_type_de.loc[~(cell_type_de[gene_id_col].isin(dups) | cell_type_de[gene_id_col].isna()),:]
cell_type_de[gene_id_col] = cell_type_de[gene_id_col].astype(int).astype(str)
cell_type_de.set_index(gene_id_col, inplace=True)

In [64]:
cell_type_de[["score"]].isna().any()

score    False
dtype: bool

In [70]:
import rpy2.robjects as robjects
from rpy2.robjects import default_converter

robjects.conversion.set_default_converter(default_converter) 

AttributeError: module rpy2.robjects.conversion has no attribute set_default_converter

In [72]:
%%R
fgseaRes_react <- fgsea(pathways = pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)

python(94524) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94525) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94526) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94527) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94528) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94529) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94530) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94531) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94532) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94533) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [74]:
%%R
colnames(fgseaRes_react)

[1] "pathway"     "pval"        "padj"        "log2err"     "ES"         
[6] "NES"         "size"        "leadingEdge"


In [74]:
already_done = ["B_cells", "ECs", "IGFBP2_cells", "Mesothelial"]

for cell_type, cell_type_de in de_res_by_ct.items():
    
    if cell_type in already_done:
        continue
    
    cell_type_de["score"] = cell_type_de["stat"]
    vc = cell_type_de.value_counts(gene_id_col)
    dups = vc[vc > 1].index
    
    cell_type_de = cell_type_de.loc[~(cell_type_de[gene_id_col].isin(dups) | cell_type_de[gene_id_col].isna()),:]
    cell_type_de[gene_id_col] = cell_type_de[gene_id_col].astype(int).astype(str)
    cell_type_de.set_index(gene_id_col, inplace=True)
    
    with conversion.localconverter(default_converter + numpy2ri.converter
         + anndata2ri.converter):
        ro.globalenv['ranks'] = cell_type_de.loc[:,"score"]

    with conversion.localconverter(default_converter):
        %R -i out_dir,cell_type setwd(out_dir)
        
        
        %R fgseaRes_react <- fgsea(pathways = pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_react, paste0(cell_type, ".reactome.csv"), row.names = FALSE)
        
        %R fgseaRes_hallmark <- fgsea(pathways = hallmark.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_hallmark, paste0(cell_type, ".hallmark.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_wp <- fgsea(pathways = wiki.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_wp, paste0(cell_type, ".wiki.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_kegg <- fgsea(pathways = kegg_pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_kegg, paste0(cell_type, ".kegg.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_bp <- fgsea(pathways = bp.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_bp, paste0(cell_type, ".bp.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_cc <- fgsea(pathways = cc.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_cc, paste0(cell_type, ".cc.csv"), row.names = FALSE)
        
        %R -i cell_type fgseaRes_mf <- fgsea(pathways = mf.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
        %R fwrite(fgseaRes_mf, paste0(cell_type, ".mf.csv"), row.names = FALSE)
    
        %R n_top <- 15
        %R p1 <- plot_fgsea_top(fgseaRes_hallmark, hallmark.pathways, ranks, n_top)
        %R p2 <- plot_fgsea_top(fgseaRes_react, pathways, ranks, n_top)
        %R p3 <- plot_fgsea_top(fgseaRes_wp, wiki.pathways, ranks, n_top)
        %R p4 <- plot_fgsea_top(fgseaRes_kegg, kegg_pathways, ranks, n_top)
        %R p5 <- plot_fgsea_top(fgseaRes_bp, bp.pathways, ranks, n_top)
        
    
        ro.r('''
        library(grid)
        
        plots <- list(p1, p2, p3, p4, p5)
        
        pdf(paste0(cell_type, ".pdf"), width = 16, height = 16)
        for (i in seq_along(plots)) {
            # if (i > 1) grid.newpage()
            grid.draw(plots[[i]])
        }
        dev.off()
        ''')

python(94575) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94576) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94577) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94578) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94579) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94580) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94581) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94582) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94583) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(94584) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


TypeError: 'NULLType' object is not iterable

## edgeR

In [155]:
t = sc.read_h5ad("/Users/willtrim/Documents/projs/Wills_analysis/obesity_scRNAseq/inputs/anndatas/omentum_reannotated_finely_pb_cell_type2_.h5ad")

In [156]:
t.var

,gene_id,entrez_id,entrez_name,description
entrez_id,,,,
7494,ENSG00000100219,7494,XBP1,X-box binding protein 1 [Source:HGNC Symbol;Ac...
4856,ENSG00000136999,4856,CCN3,cellular communication network factor 3 [Sourc...
148362,ENSG00000162819,148362,BROX,BRO1 domain and CAAX motif containing [Source:...
80024,ENSG00000089060,80024,SLC8B1,solute carrier family 8 member B1 [Source:HGNC...
91107,ENSG00000132481,91107,TRIM47,tripartite motif containing 47 [Source:HGNC Sy...
...,...,...,...,...
7543,ENSG00000005889,7543,ZFX,zinc finger protein X-linked [Source:HGNC Symb...
4232,ENSG00000106484,4232,MEST,mesoderm specific transcript [Source:HGNC Symb...
23047,ENSG00000083642,23047,PDS5B,PDS5 cohesin associated factor B [Source:HGNC ...


In [151]:
cell_type_col = "cell_type2_"
de_res = pd.read_csv(
    os.path.join(
        RES_DIR,
        f"edger_results_per_{cell_type_col}.csv"),
    # index_col=0
)
de_res.head()

,logFC,logCPM,F,PValue,FDR,cell_type,ensembl_id,entrez_id
0,-1.622898,8.883997,39.735244,0.000039,0.032328,B_cells,ENSG00000101596,23347.0
1,0.405595,13.120773,23.394669,0.000396,0.166177,B_cells,ENSG00000229117,6171.0
2,-1.209044,8.249642,18.360203,0.001051,0.185435,B_cells,ENSG00000060138,8531.0
3,-1.427702,7.461295,17.812468,0.001103,0.185435,B_cells,ENSG00000118985,22936.0
4,-1.308667,7.511898,16.245186,0.001547,0.185435,B_cells,ENSG00000173230,2804.0


In [152]:
de_res["score"] = -np.log10(de_res["FDR"])
de_res.loc[np.isinf(de_res.score), "score"] = 300
de_res.loc[de_res.logFC < 0, "score"] = -de_res.loc[de_res.logFC < 0, "score"]

In [153]:
de_res.sort_values("FDR").head()

,logFC,logCPM,F,PValue,FDR,cell_type,ensembl_id,entrez_id,score
79462,-5.327132,8.847190,262.941211,9.455413e-12,1.292555e-08,mNK2_cell,ENSG00000173110,3310.0,-7.888551
839,1.658518,6.004686,82.993472,8.056196e-08,5.006926e-04,CD4_T_cells,ENSG00000179914,55600.0,3.300429
840,2.066035,4.339887,48.407007,2.682157e-06,8.334804e-03,CD4_T_cells,ENSG00000211677,NaN,2.079105
16935,2.110200,2.705424,79.667022,1.472813e-06,8.398714e-03,FAPs,ENSG00000211679,NaN,2.075787
16936,1.735673,3.611445,84.841634,1.297665e-06,8.398714e-03,FAPs,ENSG00000211677,NaN,2.075787


In [ ]:
de_res["ensembl_id"] = adata_orig.var.loc[de_res.index, "gene_ids"].values

In [ ]:
de_res["entrez_id"] = None
idx = de_res["ensembl_id"].isin(entrez_gene_ids.index)
de_res.loc[idx, "entrez_id"] = entrez_gene_ids.loc[de_res.loc[idx, "ensembl_id"], "converted_alias"].values

In [154]:
de_res.to_csv(
    os.path.join(
        RES_DIR,
        f"edger_results_per_{cell_type_col}.csv"),
    index=False
)

In [39]:
de_res_by_ct = dict(list(de_res.groupby('cell_type')))

In [40]:
de_res_by_ct["Mesothelial"]

,variable,baseMean,log_fc,lfcSE,stat,p_value,adj_p_value,contrast,cell_type,ensembl_id,entrez_id
0,LYZ,39.173049,-1.648288,0.269394,-6.118509,9.445466e-10,0.000010,NaN,Mesothelial,ENSG00000090382,4069.0
1,GZMB,21.675012,-2.226017,0.365557,-6.089381,1.133483e-09,0.000010,NaN,Mesothelial,ENSG00000100453,3002.0
2,IGLC2,32.244079,1.969933,0.334969,5.880938,4.079490e-09,0.000023,NaN,Mesothelial,ENSG00000211677,NaN
3,PDZK1IP1,418.916476,0.906541,0.169227,5.356952,8.463775e-08,0.000364,NaN,Mesothelial,ENSG00000162366,10158.0
4,MPRIP-AS1,24.830294,-2.018823,0.389383,-5.184673,2.163944e-07,0.000745,NaN,Mesothelial,ENSG00000225442,NaN
...,...,...,...,...,...,...,...,...,...,...,...
17571,PCK1,447.314482,1.889813,1.339761,1.410560,NaN,NaN,NaN,Mesothelial,ENSG00000124253,5105.0
17572,CLDN5,14.775053,2.425018,1.585904,1.529107,NaN,NaN,NaN,Mesothelial,ENSG00000184113,7122.0
17573,HMOX1,107.798478,2.703812,1.232615,2.193558,NaN,NaN,NaN,Mesothelial,ENSG00000100292,3162.0
17574,PWWP3B,171.176119,1.611988,0.834273,1.932206,NaN,NaN,NaN,Mesothelial,ENSG00000157502,139221.0


In [46]:
out_dir = os.path.join(RES_DIR, "fgsea", cell_type_col, "deseq2")
os.makedirs(out_dir, exist_ok=True)

In [50]:
%%R
library(data.table)

plot_fgsea_top <- function(fgseaRes, pathways, ranks, n_top) {
    topPathwaysUp <- fgseaRes[ES > 0][head(order(pval), n=n_top), pathway]
    topPathwaysDown <- fgseaRes[ES < 0][head(order(pval), n=n_top), pathway]
    topPathways <- c(topPathwaysUp, rev(topPathwaysDown))
    plotGseaTable(pathways[topPathways], ranks, fgseaRes, gseaParam = 0.5)
}

data.table 1.17.8 using 1 threads (see ?getDTthreads).  Latest news: r-datatable.com
**********
This installation of data.table has not detected OpenMP support. It should still work but in single-threaded mode.
This is a Mac. Please read https://mac.r-project.org/openmp/. Please engage with Apple and ask them for support. Check r-datatable.com for updates, and our Mac instructions here: https://github.com/Rdatatable/data.table/wiki/Installation. After several years of many reports of installation problems on Mac, it's time to gingerly point out that there have been no similar problems on Windows or Linux.
**********

Attaching package: ‘data.table’

The following object is masked from ‘package:purrr’:

    transpose

The following objects are masked from ‘package:dplyr’:

    between, first, last



In [53]:
cell_type

'FAPs'

In [54]:
cell_type_de.loc[:,"score"]

entrez_id
3123    -5.901214
4069    -5.548287
51065    5.394693
23114   -5.283129
6279    -4.867909
           ...   
3778    -0.447592
6490     1.161309
9244     0.215611
3929     1.914940
30848    1.487380
Name: score, Length: 14663, dtype: float64

In [56]:
cell_type_de = de_res_by_ct["FAPs"]

In [57]:
cell_type_de["score"] = cell_type_de["stat"]
vc = cell_type_de.value_counts(gene_id_col)
dups = vc[vc > 1].index


In [60]:
cell_type_de = cell_type_de.loc[~(cell_type_de[gene_id_col].isin(dups) | cell_type_de[gene_id_col].isna()),:]
cell_type_de[gene_id_col] = cell_type_de[gene_id_col].astype(int).astype(str)
cell_type_de.set_index(gene_id_col, inplace=True)

In [64]:
cell_type_de[["score"]].isna().any()

score    False
dtype: bool

In [ ]:
already_done = ["B_cells", "ECs", "FAPs"]

for cell_type, cell_type_de in de_res_by_ct.items():
    
    if cell_type in already_done:
        continue
    
    cell_type_de["score"] = cell_type_de["stat"]
    vc = cell_type_de.value_counts(gene_id_col)
    dups = vc[vc > 1].index
    
    cell_type_de = cell_type_de.loc[~(cell_type_de[gene_id_col].isin(dups) | cell_type_de[gene_id_col].isna()),:]
    cell_type_de[gene_id_col] = cell_type_de[gene_id_col].astype(int).astype(str)
    cell_type_de.set_index(gene_id_col, inplace=True)
    
    with conversion.localconverter(default_converter + numpy2ri.converter
         + anndata2ri.converter):
        ro.globalenv['ranks'] = cell_type_de.loc[:,"score"]
    
    %R -i out_dir,cell_type setwd(out_dir)
    
    
    %R -i cell_type fgseaRes_react <- fgsea(pathways = pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_react, paste0(cell_type, ".reactome.csv"), row.names = FALSE)
    
    %R fgseaRes_hallmark <- fgsea(pathways = hallmark.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_hallmark, paste0(cell_type, ".hallmark.csv"), row.names = FALSE)
    
    %R -i cell_type fgseaRes_wp <- fgsea(pathways = wiki.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_wp, paste0(cell_type, ".wiki.csv"), row.names = FALSE)
    
    %R -i cell_type fgseaRes_kegg <- fgsea(pathways = kegg_pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_kegg, paste0(cell_type, ".kegg.csv"), row.names = FALSE)
    
    %R -i cell_type fgseaRes_bp <- fgsea(pathways = bp.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_bp, paste0(cell_type, ".bp.csv"), row.names = FALSE)
    
    %R -i cell_type fgseaRes_cc <- fgsea(pathways = cc.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_cc, paste0(cell_type, ".cc.csv"), row.names = FALSE)
    
    %R -i cell_type fgseaRes_mf <- fgsea(pathways = mf.pathways, stats = ranks, minSize  = 15, maxSize  = 500, eps=0)
    %R fwrite(fgseaRes_mf, paste0(cell_type, ".mf.csv"), row.names = FALSE)

    %R n_top <- 15
    %R p1 <- plot_fgsea_top(fgseaRes_hallmark, hallmark.pathways, ranks, n_top)
    %R p2 <- plot_fgsea_top(fgseaRes_react, pathways, ranks, n_top)
    %R p3 <- plot_fgsea_top(fgseaRes_wp, wiki.pathways, ranks, n_top)
    %R p4 <- plot_fgsea_top(fgseaRes_kegg, kegg_pathways, ranks, n_top)
    %R p5 <- plot_fgsea_top(fgseaRes_bp, bp.pathways, ranks, n_top)
    

    ro.r('''
    library(grid)
    
    plots <- list(p1, p2, p3, p4, p5)
    
    pdf(paste0(cell_type, ".pdf"), width = 16, height = 16)
    for (i in seq_along(plots)) {
        # if (i > 1) grid.newpage()
        grid.draw(plots[[i]])
    }
    dev.off()
    ''')

python(93700) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93701) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93702) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93703) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93704) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93705) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93706) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93707) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93708) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93709) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In addition: Warning message:
In preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam,  :
  There are ties in the preranked stats (0.34% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.


python(93710) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93711) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93712) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93713) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93714) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93715) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93716) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93717) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93718) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In addition: Warning message:
In preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam,  :
  There are ties in the preranked stats (0.34% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.


python(93719) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93720) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93721) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93722) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93723) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93725) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93726) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93727) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93728) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

In addition: Warning message:
In preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam,  :
  There are ties in the preranked stats (0.34% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.


python(93752) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93753) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93754) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93755) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93756) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93757) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93758) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93759) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93760) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In addition: Warning message:
In preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam,  :
  There are ties in the preranked stats (0.34% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.


python(93761) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93762) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93763) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93764) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93765) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93766) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93767) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93768) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93769) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93770) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In addition: Warning message:
In preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam,  :
  There are ties in the preranked stats (0.34% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.
